In [1]:
import os
import sys
import httpx
from dotenv import load_dotenv
load_dotenv(os.path.join("..", ".env"))

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import InjectedToolArg, InjectedToolCallId, tool
from langgraph.prebuilt import InjectedState
from langgraph.types import Command
from markdownify import markdownify
from pydantic import BaseModel, Field
from tavily import TavilyClient
from typing_extensions import Annotated, Literal

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

import os
from datetime import datetime
import uuid, base64

# Add path to unified_retriever
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'pawsitive_v1_workflow/textbook_to_db'))
from unified_retriever import UnifiedRetriever

In [16]:
# Init Sub_agent models, Retriever, Tavily client   
summarization_model = init_chat_model(model="ollama:llama3.2:3b", temperature=0.2)
tavily_client = TavilyClient(api_key="" + os.getenv("TAVILY_API_KEY"))

# Initialize Chroma retriever for textbook
persist_directory = '../chroma/Cat_Owners_Home_Veterinary_Handbook'
id_key = "doc_id"

text_embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

text_vectorstore = Chroma(
    collection_name="text_summaries_and_tables_and_image_summaries",
    embedding_function=text_embeddings,
    persist_directory=persist_directory
)

text_docstore = Chroma(
    collection_name="text_originals",
    embedding_function=text_embeddings,
    persist_directory=persist_directory
)

retriever = UnifiedRetriever(
    text_vectorstore, 
    text_docstore, 
    id_key=id_key
)

class Summary(BaseModel):
    """Schema for webpage content summarization."""
    filename: str = Field(description="Name of the file to store.")
    summary: str = Field(description="Key learnings from the webpage.")

def get_today_str() -> str:
    """Get current date in a human-readable format."""
    return datetime.now().strftime("%a %b %-d, %Y")

def run_tavily_search(
    search_query: str, 
    max_results: int = 1, 
    topic: Literal["general", "news", "finance"] = "general", 
    include_raw_content: bool = True, 
) -> dict:
    
    result = tavily_client.search(
        query=search_query,
        search_depth="advanced",
        max_results=max_results,
        include_domains=[
            "https://www.avma.org/",
            "https://www.merckvetmanual.com/",
            "https://www.vet.cornell.edu/",
            "https://www.vetmed.ucdavis.edu/",
            "https://vcahospitals.com/know-your-pet",
            "https://www.petmd.com/",
            "https://www.aspca.org/pet-care",
            "https://www.petpoisonhelpline.com/",
            "https://www.wormsandgermsblog.com/",
            "https://www.vin.com/"
            ],
        exclude_domains=["reddit.com", "quora.com","wikipedia.org"],
        include_raw_content=True
    )

    return result

from tools_and_prompts.prompts import SUMMARIZE_WEB_SEARCH
def summarize_webpage_content(webpage_content: str) -> Summary:
    try:
        # Set up structured output model for summarization
        structured_model = summarization_model.with_structured_output(Summary)
        
        # Generate summary
        summary_and_filename = structured_model.invoke([
            HumanMessage(content=SUMMARIZE_WEB_SEARCH.format(
                webpage_content=webpage_content, 
                date=get_today_str()
            ))
        ])
        
        return summary_and_filename
        
    except Exception:
        # Return a basic summary object on failure
        return Summary(
            filename="search_result.md",
            summary=webpage_content[:1000] + "..." if len(webpage_content) > 1000 else webpage_content
        )


def process_search_results(results: dict) -> list[dict]:
    processed_results = []
    HTTPX_CLIENT = httpx.Client()
    
    for result in results.get('results', []):
        
        url = result['url']
        response = HTTPX_CLIENT.get(url)

        if response.status_code == 200:
            # Convert HTML to markdown
            raw_content = markdownify(response.text)
            summary_obj = summarize_webpage_content(raw_content)
        else:
            # Use Tavily's generated summary
            raw_content = result.get('raw_content', '')
            summary_obj = Summary(
                filename="URL_error.md",
                summary=result.get('content', 'Error reading URL; try another search.')
            )
        
        # uniquify file names
        uid = base64.urlsafe_b64encode(uuid.uuid4().bytes).rstrip(b"=").decode("ascii")[:8]
        name, ext = os.path.splitext(summary_obj.filename)
        summary_obj.filename = f"{name}_{uid}{ext}"

        processed_results.append({
            'url': result['url'],
            'title': result['title'],
            'summary': summary_obj.summary,
            'filename': summary_obj.filename,
            'raw_content': raw_content,
        })
    
    return processed_results

In [15]:
# Define the Tavily tools
from tools_and_prompts.state import DeepAgentState
@tool(parse_docstring=True)
def tavily_search(
    query: str,
    state: Annotated[DeepAgentState, InjectedState],
    tool_call_id: Annotated[str, InjectedToolCallId],
    max_results: Annotated[int, InjectedToolArg] = 1,
    topic: Annotated[Literal["general", "news", "finance"], InjectedToolArg] = "general",
) -> Command:
    """Search web and save detailed results to files while returning minimal context.

    Performs web search and saves full content to files for context offloading.
    Returns only essential information to help the agent decide on next steps.

    Args:
        query: Search query to execute
        state: Injected agent state for file storage
        tool_call_id: Injected tool call identifier
        max_results: Maximum number of results to return (default: 1)
        topic: Topic filter - 'general', 'news', or 'finance' (default: 'general')

    Returns:
        Command that saves full results to files and provides minimal summary
    """
    
    search_results = run_tavily_search(
        query,
        max_results=max_results,
        topic=topic,
        include_raw_content=True,
    ) 

    
    processed_results = process_search_results(search_results)
    
    # Save each result to a file and prepare summary
    files = state.get("files", {})
    saved_files = []
    summaries = []
    
    for i, result in enumerate(processed_results):
        # Use the AI-generated filename from summarization
        filename = result['filename']
        
        # Create file content with full details
        file_content = f"""# Search Result: {result['title']}

**URL:** {result['url']}
**Query:** {query}
**Date:** {get_today_str()}

## Summary
{result['summary']}

## Raw Content
{result['raw_content'] if result['raw_content'] else 'No raw content available'}
"""
        
        files[filename] = file_content
        saved_files.append(filename)
        summaries.append(f"- {filename}: {result['summary']}...")
    
    # Create minimal summary for tool message - focus on what was collected
    summary_text = f"""🔍 Found {len(processed_results)} result(s) for '{query}':

{chr(10).join(summaries)}

Files: {', '.join(saved_files)}
💡 Use read_file() to access full details when needed."""

    return Command(
        update={
            "files": files,
            "messages": [
                ToolMessage(summary_text, tool_call_id=tool_call_id)
            ],
        }
    )

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:502: UserWarning: typing.NotRequired is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:502: UserWarning: typing.NotRequired is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(


In [ ]:
# Define the Textbook Search tool
@tool(parse_docstring=True)
def textbook_search(
    query: str,
    state: Annotated[DeepAgentState, InjectedState],
    tool_call_id: Annotated[str, InjectedToolCallId],
    k: Annotated[int, InjectedToolArg] = 3,
) -> Command:
    """Search veterinary textbook and save detailed results to files.

    Searches the Cat Owner's Home Veterinary Handbook database for relevant
    information. Saves full textbook content to files for context offloading.
    Returns only essential information to help the agent decide on next steps.

    Args:
        query: Search query to execute on the textbook database
        state: Injected agent state for file storage
        tool_call_id: Injected tool call identifier
        k: Number of results to retrieve (default: 5)

    Returns:
        Command that saves full results to files and provides minimal summary
    """
    
    # Retrieve from textbook
    results = retriever.retrieve_multi_modal(query, k=k)
    
    # Save each result to a file and prepare summary
    files = state.get("files", {})
    saved_files = []
    summaries = []
    
    for i, doc in enumerate(results):
        modality = doc.get('modality', 'text')
        doc_id = doc.get('doc_id', 'unknown')
        summary = doc.get('summary', '')
        
        # Fetch full original text from docstore
        try:
            original = retriever.text_docstore._collection.get(
                ids=[doc_id], 
                include=["documents", "metadatas"]
            )
            original_text = original["documents"][0] if original["documents"] else summary
            original_metadata = original["metadatas"][0] if original["metadatas"] else {}
        except Exception:
            original_text = summary
            original_metadata = {}
        
        # Generate filename
        uid = base64.urlsafe_b64encode(uuid.uuid4().bytes).rstrip(b"=").decode("ascii")[:8]
        filename = f"textbook_{modality}_{uid}.md"
        
        # Create file content with full details
        file_content = f"""# Textbook Search Result: {modality.upper()}

**Doc ID:** {doc_id}
**Query:** {query}
**Date:** {get_today_str()}
**Type:** {modality}

## Summary
{summary}

## Full Content
{original_text}

## Metadata
{original_metadata}
"""
        
        files[filename] = file_content
        saved_files.append(filename)
        # Truncate summary for display
        summary_preview = summary[:100] + "..." if len(summary) > 100 else summary
        summaries.append(f"- {filename}: [{modality}] {summary_preview}")
    
    # Create minimal summary for tool message
    summary_text = f"""📚 Found {len(results)} textbook result(s) for '{query}':

{chr(10).join(summaries)}

Files: {', '.join(saved_files)}
💡 Use read_file() to access full textbook content when needed."""

    return Command(
        update={
            "files": files,
            "messages": [
                ToolMessage(summary_text, tool_call_id=tool_call_id)
            ],
        }
    )

In [ ]:
# Define the Strategic Thinking tool
@tool(parse_docstring=True)
def think_tool(reflection: str) -> str:
    """Tool for strategic reflection on research progress and decision-making.

    Use this tool after each search to analyze results and plan next steps systematically.
    This creates a deliberate pause in the research workflow for quality decision-making.

    When to use:
    - After receiving search results: What key information did I find?
    - Before deciding next steps: Do I have enough to answer comprehensively?
    - When assessing research gaps: What specific information am I still missing?
    - Before concluding research: Can I provide a complete answer now?
    - How complex is the question: Have I reached the number of search limits?

    Reflection should address:
    1. Analysis of current findings - What concrete information have I gathered?
    2. Gap assessment - What crucial information is still missing?
    3. Quality evaluation - Do I have sufficient evidence/examples for a good answer?
    4. Strategic decision - Should I continue searching or provide my answer?

    Args:
        reflection: Your detailed reflection on research progress, findings, gaps, and next steps

    Returns:
        Confirmation that reflection was recorded for decision-making
    """
    return f"Reflection recorded: {reflection}"

## Deep Agent

Now, we can just apply all of our prior learnings: 

* We'll give the researcher a `think_tool` and our `search_tool` above.
* We'll give our parent agent `file tools`, a `think_tool`, and a `task` tool. 

In [ ]:
from datetime import datetime

from IPython.display import Image, display
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from utils import show_prompt, stream_agent

from tools_and_prompts.file_tools import ls, read_file, write_file
from tools_and_prompts.prompts import (
    FILE_USAGE_INSTRUCTIONS,
    RESEARCHER_INSTRUCTIONS,
    SUBAGENT_USAGE_INSTRUCTIONS,
    TODO_USAGE_INSTRUCTIONS,
)
from tools_and_prompts.state import DeepAgentState
from tools_and_prompts.task_tool import _create_task_tool

from tools_and_prompts.todo_tools import write_todos, read_todos

In [17]:
# Create agent using create_react_agent directly
# model = init_chat_model(model="anthropic:claude-sonnet-4-5-20250929",temperature=0.5)
model = init_chat_model(model="ollama:qwen3:8b",temperature=0.5)

# Limits
max_concurrent_research_units = 3
max_researcher_iterations = 3

# Tools
sub_agent_tools = [textbook_search, tavily_search, think_tool]
built_in_tools = [ls, read_file, write_file, write_todos, read_todos, think_tool]

# Create research sub-agent
research_sub_agent = {
    "name": "research-agent",
    "description": "Delegate research to the sub-agent researcher. Only give this researcher one topic at a time.",
    "prompt": RESEARCHER_INSTRUCTIONS.format(date=get_today_str()),
    "tools": ["textbook_search", "tavily_search", "think_tool"],
}

# Create task tool to delegate tasks to sub-agents
task_tool = _create_task_tool(
    sub_agent_tools, [research_sub_agent], model, DeepAgentState
)

delegation_tools = [task_tool]
all_tools = sub_agent_tools + built_in_tools + delegation_tools  # search available to main agent for trivial cases

# Build prompt
SUBAGENT_INSTRUCTIONS = SUBAGENT_USAGE_INSTRUCTIONS.format(
    max_concurrent_research_units=max_concurrent_research_units,
    max_researcher_iterations=max_researcher_iterations,
    date=datetime.now().strftime("%a %b %-d, %Y"),
)

INSTRUCTIONS = (
    "# TODO MANAGEMENT\n"
    + TODO_USAGE_INSTRUCTIONS
    + "\n\n"
    + "=" * 80
    + "\n\n"
    + "# FILE SYSTEM USAGE\n"
    + FILE_USAGE_INSTRUCTIONS
    + "\n\n"
    + "=" * 80
    + "\n\n"
    + "# SUB-AGENT DELEGATION\n"
    + SUBAGENT_INSTRUCTIONS
)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:502: UserWarning: typing.NotRequired is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:502: UserWarning: typing.NotRequired is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(


## SQLite Persistent Memory Setup

Adding **SQLite-based persistent memory** using `SqliteSaver` to enable:
- ✅ **Conversation persistence** across kernel restarts
- ✅ **Thread-based conversations** - each thread_id maintains separate conversation history
- ✅ **Disk-based storage** - all conversations saved to SQLite database
- ✅ **Production-ready** - suitable for real applications

### How it works:
- **thread_id**: Acts like a conversation channel identifier
- **Automatic state saving**: After each interaction, entire state (messages, files, todos) saved to SQLite
- **Context preservation**: Same thread_id = continues conversation; different thread_id = new conversation

In [8]:
# Import SQLite checkpointer for persistent external memory
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

# Create SQLite database connection
# Using a file-based database for persistence across kernel restarts
db_path = "agent_memory.db"
conn = sqlite3.connect(db_path, check_same_thread=False)

# Create the SQLite checkpointer
memory = SqliteSaver(conn)

print(f"✅ SQLite checkpointer created")
print(f"💾 Database location: {db_path}")
print(f"🔄 Persists across kernel restarts: Yes")

✅ SQLite checkpointer created
💾 Database location: agent_memory.db
🔄 Persists across kernel restarts: Yes


In [19]:
from utils import format_messages
from IPython.display import Image, display

agent = create_react_agent(
    model, 
    all_tools, 
    prompt=INSTRUCTIONS, 
    state_schema=DeepAgentState,
    checkpointer=memory 
)

# Show the agent graph
# display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

# Verify checkpointer is attached
print(f"\n✅ Checkpointer attached: {agent.checkpointer is not None}")
print(f"📊 Checkpointer type: {type(agent.checkpointer).__name__}")
print(f"💾 Database location: {db_path}")
print(f"🔄 Persists across kernel restarts: Yes")

# Configure thread for conversation
config = {"configurable": {"thread_id": "5"}}

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "I think my cat is in pain, what would I do?",
            }
        ],
    },
    config=config  
)

format_messages(result["messages"])


✅ Checkpointer attached: True
📊 Checkpointer type: SqliteSaver
💾 Database location: agent_memory.db
🔄 Persists across kernel restarts: Yes


╭─────────────────────────────────────────────────── 🧑 Human ────────────────────────────────────────────────────╮
│ I think my cat is in pain, what would I do?                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│ 🔧 Tool Call: write_todos                                                                                       │
│    Args: {                                                                                                      │
│   "todos": [                                                                                                    │
│     {                                                                                                           │
│       "content": "Research common signs of cat pain and first aid steps",                                       │
│       "status": "pending"                                                                                       │
│     },                                                                                                          │
│     {                                                                                                           │
│       "content": "Check for existing TODOs",                                                                    │
│       "status": "pending"                                                                                       │
│     }                                                                                                           │
│   ]                                                                                                             │
│ }                                                                                                               │
│    ID: 01bba8cc-fd34-4699-b14c-1d3d1d7bc42d                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ Updated todo list to [{'content': 'Research common signs of cat pain and first aid steps', 'status':            │
│ 'pending'}, {'content': 'Check for existing TODOs', 'status': 'pending'}]                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│ 🔧 Tool Call: ls                                                                                                │
│    Args: {}                                                                                                     │
│    ID: 6d451002-887e-4a39-831f-d1a2f56bf4d2                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│ 🔧 Tool Call: read_file                                                                                         │
│    Args: {                                                                                                      │
│   "file_path": "/textbook_text_gZEXNzOT.md",                                                                    │
│   "limit": 2000                                                                                                 │
│ }                                                                                                               │
│    ID: 288ec130-d855-428b-8a78-8be63aa5355c                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ Error: File '/textbook_text_gZEXNzOT.md' not found                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│ 🔧 Tool Call: tavily_search                                                                                     │
│    Args: {                                                                                                      │
│   "query": "Common signs of cat pain and first aid steps for cat injuries"                                      │
│ }                                                                                                               │
│    ID: 7bfe7771-a99e-4234-b1e3-04c695eda38c                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ Error: AttributeError("'NoneType' object has no attribute 'filename'")                                          │
│  Please fix your mistakes.                                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│ I'll provide guidance on what to do if your cat is in pain, based on standard veterinary care principles:       │
│                                                                                                                 │
│ 1. **Observe and Document**:                                                                                    │
│    - Note any changes in behavior (lethargy, aggression, withdrawal)                                            │
│    - Check for physical signs: limping, swelling, discharge, or difficulty moving                               │
│                                                                                                                 │
│ 2. **Immediate Actions**:                                                                                       │
│    - Create a calm, quiet environment                                                                           │
│    - Avoid handling the cat if they show distress                                                               │
│    - Keep the cat warm with a blanket                                                                           │
│                                                                                                                 │
│ 3. **Emergency Signs (Seek vet immediately)**:                                                                  │
│    - Difficulty breathing                                                                                       │
│    - Vomiting or diarrhea                                                                                       │
│    - Unconsciousness                                                                                            │
│    - Severe swelling or deformity                                                                               │
│    - Persistent crying or aggression                                                                            │
│                                                                                                                 │
│ 4. **Pain Management (Do NOT give human medications)**:                                                         │
│    - Never administer aspirin, ibuprofen, or other human painkillers                                            │
│    - Use only veterinary-approved medications prescribed by a vet                                               │
│                                                                                                                 │
│ 5. **When to See a Vet**:                                                                                       │
│    - Pain persists for more than 24 hours                                                                       │
│    - Any suspected injury or illness                                                                            │
│    - Changes in appetite/behavior                                                                               │
│    - Pregnancy or nursing cats showing distress                                                                 │
│                                                                                                                 │
│ Would you like me to help you create a checklist of observations to share with your veterinarian?               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Continue Conversation with Memory

Using the same `thread_id="1"` will continue the conversation. The agent remembers all previous messages!

In [ ]:
# Continue conversation with the same thread_id
config = {"configurable": {"thread_id": "2"}}

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """I copy and paste this cat food's info, can you take a look for me? To see if it's a good choice for my cat? 
                INGREDIENTS: Turkey, turkey meal, trout, herring meal, spray dried egg product, sweet potato, buckwheat, lentils, pumpkin, peas, natural flavour, flaxseeds, chicken fat (preserved with mixed tocopherols and citric acid), sun-dried miscanthus grass, lecithin, blueberries, cranberries, apples, spinach, calcium propionate, sodium tripolyphosphate, DL-methionine, yeast extract, chicory root extract, taurine, choline chloride, salt, dried kelp, L-lysine, L-tryptophan, fumaric acid, lactic acid, citric acid, malic acid, yucca schidigera extract, thyme extract, zinc oxide, ferrous sulfate, zinc amino acid complex, iron amino acid complex, niacinamide, manganous oxide, vitamin E supplement, vitamin A supplement, calcium pantothenate, riboflavin, thiamine mononitrate, pyridoxine hydrochloride, copper sulfate, copper proteinate, manganese amino acid complex, folic acid, vitamin D3 supplement, vitamin B12 supplement, biotin, calcium iodate, sodium selenite, rosemary extract.

Guaranteed Analysis
GUARANTEED ANALYSIS: Crude Protein (min) 36%, Crude Fat (min) 16%, Crude Fiber (max) 6%, Moisture (max) 10%, Ash (max) 9%, Calcium (min) 1.8%, Phosphorous (min) 1.1%, Magnesium (max) 0.12%, Zinc (min) 100mg/kg, Omega-3 Fatty Acids (min)* 1.3%, Omega-6 Fatty Acids (min)* 2.8%
*Not recognized as an essential nutrient by the AAFCO CAT Food Nutrient Profiles.
CALORIE CONTENT (Calculated ME): 3492 kcal/kg - 384 kcal/cup
                
                """,
            }
        ],
    },
    config=config
)

format_messages(result["messages"])

## Start a New Conversation

Using a different `thread_id` starts a completely new conversation. The agent won't remember anything from the previous thread.

In [ ]:
# New thread = new conversation (agent won't remember previous discussion)
new_config = {"configurable": {"thread_id": "2"}}

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What were we just talking about?",
            }
        ],
    },
    config=new_config
)

format_messages(result["messages"])

## Inspecting Saved State

You can inspect the saved conversation state at any time using `agent.get_state()`:

In [13]:
# Get the saved state for thread_id="1"
saved_state = agent.get_state({"configurable": {"thread_id": "4"}})
print(f"Number of messages: {len(saved_state.values.get('messages', []))}")
print(f"Files created: {list(saved_state.values.get('files', {}).keys())}")
print(f"Todos: {saved_state.values.get('todos', [])}")

Number of messages: 29
Files created: ['user_request.md', 'textbook_text_iNOT5qUi.md', 'textbook_image_summary_OByiu1ci.md', 'textbook_text_9q8Nyw6U.md', 'textbook_text_Gf4pFVI7.md', 'textbook_text_cZbCRiHz.md', 'textbook_text_VGM8XECL.md', 'textbook_text_FKTOClDk.md', 'textbook_text_U9FZ2E9_.md', 'textbook_image_summary_hJ4eyDA0.md', 'textbook_text_gJRSMcgR.md', 'textbook_image_summary_HXhgdaXX.md', 'textbook_text_gpYp34UH.md', 'textbook_text_NfMFh4eG.md', 'textbook_text_3CIsK25L.md', 'textbook_image_summary_qy74wwLR.md', 'textbook_text_Qo3LMF-8.md', 'textbook_text_Byyz5e_p.md', 'textbook_text_M05HGG_s.md', 'textbook_table_aUCxZprK.md', 'textbook_text_3aKhswHR.md', 'URL_error_8BmcM0Rc.md', 'URL_error_thi4MEOO.md', 'URL_error_24DcVztT.md', 'textbook_text_N7PgjUeV.md', 'textbook_text_inTHMUl7.md', 'textbook_text_XQqXTOBW.md', 'textbook_text_dic9Hrl9.md', 'textbook_text_Z-1FNYrn.md', 'textbook_image_summary_McS2yuu0.md', 'textbook_text_8h5d2bsa.md', 'textbook_text_zHxqmz-H.md', 'textbook

## Inspecting the SQLite Database

You can also directly query the SQLite database to see all saved threads and checkpoints:

In [ ]:
# Query the database to see all saved threads
cursor = conn.cursor()

# Check database tables
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("🗄️  Database tables:", tables)

# See how many checkpoints (conversation states) are saved
cursor.execute("SELECT COUNT(*) FROM checkpoints;")
checkpoint_count = cursor.fetchone()[0]
print(f"📝 Total checkpoints saved: {checkpoint_count}")

# See all unique thread_ids
cursor.execute("SELECT DISTINCT thread_id FROM checkpoints;")
threads = cursor.fetchall()
print(f"🧵 Active threads: {[t[0] for t in threads]}")